In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.tools import tool
from langchain_groq import ChatGroq

In [4]:
model = ChatGroq(
    model="llama-3.1-8b-instant", 
    temperature = 0.7
)

In [5]:
@tool
def search_web(query: str) -> str:
    """Search the web for information."""
    return f"Search results for: {query}"

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {to} with subject: {subject}"

@tool
def delete_records(table: str, condition: str) -> str:
    """Delete records from the database."""
    return f"Deleted records from {table} where {condition}"


In [7]:

# Create agent with HITL middleware
hitl_agent = create_agent(
    model=model,
    tools=[search_web, send_email, delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,       # Require approval
                "delete_records": True,   # Require approval
                "search_web": False,      # Auto-approve
            }
        ),
    ],
    checkpointer=InMemorySaver(),  # Required for state persistence
)

print("Human-in-the-Loop agent created!")

Human-in-the-Loop agent created!


In [8]:
# Step 1: Invoke — agent will pause before send_email
config = {"configurable": {"thread_id": "session_001"}}

result = hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Send an email to team@company.com about the Q4 results"}]},
    config=config
)

print("=== Agent paused — awaiting human approval ===")
print(result)

=== Agent paused — awaiting human approval ===
{'messages': [HumanMessage(content='Send an email to team@company.com about the Q4 results', additional_kwargs={}, response_metadata={}, id='0e0449af-c7b9-4e5e-a449-39518b77945d'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'eh7khhn9t', 'function': {'arguments': '{"body":"The Q4 results are in. Please find the details below.","subject":"Q4 Results","to":"team@company.com"}', 'name': 'send_email'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 347, 'total_tokens': 390, 'completion_time': 0.059045144, 'completion_tokens_details': None, 'prompt_time': 0.022984249, 'prompt_tokens_details': None, 'queue_time': 0.055303721, 'total_time': 0.082029393}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fec03-7b67-7173-851b-fd32c2a

In [11]:
approved_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config   # Same thread_id resumes the paused session
)

print("=== Approved! Final response ===")
print(approved_result["messages"][-1].content)
print(approved_result)

=== Approved! Final response ===

{'messages': [HumanMessage(content='Send an email to team@company.com about the Q4 results', additional_kwargs={}, response_metadata={}, id='0e0449af-c7b9-4e5e-a449-39518b77945d'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'eh7khhn9t', 'function': {'arguments': '{"body":"The Q4 results are in. Please find the details below.","subject":"Q4 Results","to":"team@company.com"}', 'name': 'send_email'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 347, 'total_tokens': 390, 'completion_time': 0.059045144, 'completion_tokens_details': None, 'prompt_time': 0.022984249, 'prompt_tokens_details': None, 'queue_time': 0.055303721, 'total_time': 0.082029393}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fec03-7b67-7173-851b-fd32c2a134aa-0', too

In [12]:
# Step 1: Invoke — agent will pause before send_email
config2 = {"configurable": {"thread_id": "session_002"}}

result = hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Delete all the records from the user database"}]},
    config=config2
)

print("=== Agent paused — awaiting human approval ===")
print(result)

=== Agent paused — awaiting human approval ===
{'messages': [HumanMessage(content='Delete all the records from the user database', additional_kwargs={}, response_metadata={}, id='9fecce21-18ee-48ed-a3b7-4bff584fd80e'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'hq8pv9jyc', 'function': {'arguments': '{"condition":"","table":"user_database"}', 'name': 'delete_records'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 342, 'total_tokens': 361, 'completion_time': 0.032849018, 'completion_tokens_details': None, 'prompt_time': 0.029166101, 'prompt_tokens_details': None, 'queue_time': 0.067585789, 'total_time': 0.062015119}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fec07-a28c-7341-94e9-06db79d0e9e2-0', tool_calls=[{'name': 'delete_records', 'args': {'condition': '', '

In [13]:
approved_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "reject" , "reason":"Too risky, Requires DBA Reveiw"}]}),
    config=config2   # Same thread_id resumes the paused session
)

print("=== Approved! Final response ===")
print(approved_result["messages"][-1].content)
print(approved_result)

=== Approved! Final response ===
I can't help with deleting records from a database. 

Is there something else I can help you with?
{'messages': [HumanMessage(content='Delete all the records from the user database', additional_kwargs={}, response_metadata={}, id='9fecce21-18ee-48ed-a3b7-4bff584fd80e'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'hq8pv9jyc', 'function': {'arguments': '{"condition":"","table":"user_database"}', 'name': 'delete_records'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 342, 'total_tokens': 361, 'completion_time': 0.032849018, 'completion_tokens_details': None, 'prompt_time': 0.029166101, 'prompt_tokens_details': None, 'queue_time': 0.067585789, 'total_time': 0.062015119}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fec07-a28c-7341-94e